# Run-to-run variance [Step 08.04 - Is that improvement real?]

> **MLCourse - Agentic AI - Agent Patterns**

You change a prompt. The benchmark goes from 0.75 to 1.00. Did you improve the agent?

You cannot know until you know **how much the score moves when you change nothing**.
That is run-to-run variance, and measuring it is the difference between an
evaluation and a superstition.

### What you'll learn

- Measuring the noise floor by repeating an identical run.
- Which tasks are **flaky** vs **reliably passing/failing**, and why the per-task view
  matters more than the mean.
- Turning variance into an error bar, and the smallest difference your benchmark can
  actually detect.

### Key takeaways

- Report **mean +/- spread over R runs**, never a single number.
- A difference smaller than your noise floor is **not a result**, however much you
  want it to be.
- Flaky tasks dominate the variance. Find them and you know where to look.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 3.5
print("PACE =", PACE)

PACE = 3.5


### The benchmark: world, tools, tasks, grader


In [ ]:
# Four pieces, and they must be separable. If the grader lives inside the agent,
# you are not running a benchmark, you are running a demo.

import re

# 1. THE WORLD -----------------------------------------------------------------
EMPLOYEES = {
    "priya":  dict(name="Priya",  dept="Engineering", salary=82000),
    "marco":  dict(name="Marco",  dept="Engineering", salary=61000),
    "ana":    dict(name="Ana",    dept="Design",      salary=74000),
    "kofi":   dict(name="Kofi",   dept="Engineering", salary=69000),
    "lena":   dict(name="Lena",   dept="Design",      salary=71000),
}

TOOL_CALLS = {"n": 0}


# 2. THE TOOLS -----------------------------------------------------------------
def tool_lookup(arg):
    """lookup(name) -> the employee record, or an error string."""
    TOOL_CALLS["n"] += 1
    rec = EMPLOYEES.get(arg.strip().lower().strip('"\''))
    if not rec:
        return "ERROR: no employee named %r" % arg
    return "name=%s dept=%s salary=%d" % (rec["name"], rec["dept"], rec["salary"])


def tool_count_dept(arg):
    """count_dept(department) -> how many people work in it."""
    TOOL_CALLS["n"] += 1
    d = arg.strip().lower().strip('"\'')
    n = sum(1 for r in EMPLOYEES.values() if r["dept"].lower() == d)
    return "count=%d" % n


TOOLS = {"lookup": tool_lookup, "count_dept": tool_count_dept}

TOOL_DOC = """You have exactly two tools:
  lookup(name)            -> name, department and salary of one employee
  count_dept(department)  -> how many people work in that department

To use a tool, reply with ONLY one line:
CALL: <tool_name>(<argument>)

When you know the answer, reply with ONLY one line:
FINAL: <answer>

Never output both. Never explain. One line per turn."""


# 3. THE TASKS -----------------------------------------------------------------
# Each task carries its OWN grader. Different tasks need different checks, and
# pretending otherwise is how benchmarks end up measuring string formatting.

def num_grader(expected, tol=0.5):
    def check(answer):
        nums = re.findall(r"-?\d+(?:\.\d+)?", (answer or "").replace(",", ""))
        if not nums:
            return False
        return abs(float(nums[-1]) - expected) <= tol
    return check


def word_grader(expected):
    def check(answer):
        return expected.lower() in (answer or "").lower()
    return check


TASKS = [
    dict(id="salary",   q="What is Priya's salary?",
         grade=num_grader(82000), min_tools=1),
    dict(id="headcount", q="How many people work in Engineering?",
         grade=num_grader(3), min_tools=1),
    dict(id="combined", q="What is the combined salary of Priya and Marco? Give a single number.",
         grade=num_grader(143000), min_tools=2),
    dict(id="compare",  q="Who earns more, Ana or Marco? Answer with only the name.",
         grade=word_grader("ana"), min_tools=2),
]

print("%d tasks, %d tools, %d employees" % (len(TASKS), len(TOOLS), len(EMPLOYEES)))


### The agent under test


In [ ]:
# A minimal ReAct-shaped loop: the model either calls a tool or answers. Capped at
# MAX_STEPS, because an unbounded agent in a benchmark is an unbounded bill.

MAX_STEPS = 3
CALL_RE = re.compile(r"CALL:\s*(\w+)\s*\((.*?)\)", re.S)
FINAL_RE = re.compile(r"FINAL:\s*(.+)", re.S)


def run_agent(question, temperature=0.0, verbose=False):
    """Run one attempt. Returns a dict - never raises, so one bad task cannot
    abort a benchmark run halfway through."""
    llm = make_llm(temperature=temperature, max_tokens=120)
    TOOL_CALLS["n"] = 0
    transcript = []
    messages = [("system", "You are a precise data assistant.\n\n" + TOOL_DOC),
                ("user", question)]
    tin = tout = 0
    answer = None
    steps = 0

    for steps in range(1, MAX_STEPS + 1):
        msg = safe_invoke(llm, messages)
        u = msg.usage_metadata or {}
        tin += u.get("input_tokens", 0)
        tout += u.get("output_tokens", 0)
        text = msg.content.strip()
        transcript.append(text)
        if verbose:
            print("  [step %d] %s" % (steps, text.replace("\n", " ")[:100]))

        fin = FINAL_RE.search(text)
        call = CALL_RE.search(text)
        # A model that emits both is ambiguous; prefer the tool call, since a
        # premature FINAL is the more common failure.
        if call and (not fin or call.start() < fin.start()):
            tool, arg = call.group(1), call.group(2)
            result = TOOLS[tool](arg) if tool in TOOLS else "ERROR: no such tool %r" % tool
            if verbose:
                print("           -> %s" % result)
            messages = messages + [("assistant", text), ("user", "TOOL RESULT: " + result)]
            continue
        if fin:
            answer = fin.group(1).strip().splitlines()[0].strip()
            break
        # Neither: nudge once, then give up.
        messages = messages + [("assistant", text),
                               ("user", "Reply with ONLY one line: CALL: ... or FINAL: ...")]

    return dict(answer=answer, steps=steps, tool_calls=TOOL_CALLS["n"],
                tokens=tin + tout, tin=tin, tout=tout, transcript=transcript)


### 1. Repeat the identical run

Three repeats of the same four tasks, same temperature, same prompts. Nothing
changes between runs except what the provider does on its side.

(Three is the minimum that gives you any spread at all. Real work uses 5-10; we are
sharing a rate limit.)

In [5]:
R = 3
TEMP = 0.8      # >0 so the agent can vary; at T=0 you measure only provider drift

runs = []
for rep in range(R):
    print("=" * 62)
    print("REPEAT %d/%d" % (rep + 1, R))
    rows = []
    for t in TASKS:
        r = run_agent(t["q"], temperature=TEMP)
        ok = bool(t["grade"](r["answer"]))
        rows.append(dict(id=t["id"], passed=ok, answer=r["answer"],
                         tokens=r["tokens"], tools=r["tool_calls"], steps=r["steps"]))
        print("  %-11s %-5s answer=%-22s tools=%d tokens=%d"
              % (t["id"], "PASS" if ok else "FAIL", str(r["answer"])[:22],
                 r["tool_calls"], r["tokens"]))
    runs.append(rows)

REPEAT 1/3


  [rate limit] sleeping 2.0s (attempt 1/5)


  salary      PASS  answer=82000                  tools=1 tokens=302


  headcount   PASS  answer=3                      tools=1 tokens=287


  [rate limit] sleeping 2.0s (attempt 1/5)


...

  [rate limit] sleeping 2.0s (attempt 1/5)


  combined    PASS  answer=143000                 tools=2 tokens=536


  [rate limit] sleeping 2.0s (attempt 1/5)


  [rate limit] sleeping 2.0s (attempt 1/5)


  compare     PASS  answer=Ana                    tools=2 tokens=519


In [6]:
import statistics as stats

scores = [sum(1 for r in rows if r["passed"]) / len(rows) for rows in runs]
tokens = [sum(r["tokens"] for r in rows) for rows in runs]

print("=" * 62)
print("scores per run : %s" % [round(s, 3) for s in scores])
print("mean           : %.3f" % stats.mean(scores))
print("min - max      : %.3f - %.3f  (spread %.3f)"
      % (min(scores), max(scores), max(scores) - min(scores)))
if R > 1:
    sd = stats.stdev(scores)
    print("std dev        : %.3f" % sd)
    print("REPORT AS      : %.3f +/- %.3f over %d runs, %d tasks"
          % (stats.mean(scores), sd, R, len(TASKS)))
print()
print("tokens per run : %s  (mean %d, spread %d)"
      % (tokens, int(stats.mean(tokens)), max(tokens) - min(tokens)))

scores per run : [1.0, 1.0, 1.0]
mean           : 1.000
min - max      : 1.000 - 1.000  (spread 0.000)
std dev        : 0.000
REPORT AS      : 1.000 +/- 0.000 over 3 runs, 4 tasks

tokens per run : [1644, 1644, 1644]  (mean 1644, spread 0)


### 2. Per-task stability - the useful view

The mean hides everything. Split the tasks into three groups: always passed, always
failed, and **flaky**. Only the flaky ones contribute variance, and they are where
your engineering effort belongs.

In [7]:
print("%-12s %-14s %-12s %s" % ("task", "pass pattern", "rate", "classification"))
print("-" * 64)
stable_pass = stable_fail = flaky = 0
for i, t in enumerate(TASKS):
    pattern = [rows[i]["passed"] for rows in runs]
    rate = sum(pattern) / R
    if rate == 1.0:
        cls = "stable PASS"
        stable_pass += 1
    elif rate == 0.0:
        cls = "stable FAIL  <- capability gap"
        stable_fail += 1
    else:
        cls = "FLAKY        <- source of variance"
        flaky += 1
    print("%-12s %-14s %-12s %s"
          % (t["id"], "".join("P" if p else "F" for p in pattern),
             "%d/%d" % (sum(pattern), R), cls))
print("-" * 64)
print("stable pass %d | stable fail %d | flaky %d" % (stable_pass, stable_fail, flaky))

task         pass pattern   rate         classification
----------------------------------------------------------------
salary       PPP            3/3          stable PASS
headcount    PPP            3/3          stable PASS
combined     PPP            3/3          stable PASS
compare      PPP            3/3          stable PASS
----------------------------------------------------------------
stable pass 4 | stable fail 0 | flaky 0


In [8]:
print("answers given per task across runs (flakiness usually shows up here)")
print("-" * 70)
for i, t in enumerate(TASKS):
    answers = [str(rows[i]["answer"]) for rows in runs]
    distinct = len(set(answers))
    print("%-12s %d distinct answer(s): %s" % (t["id"], distinct,
                                               " | ".join(a[:20] for a in answers)))
print("-" * 70)

answers given per task across runs (flakiness usually shows up here)
----------------------------------------------------------------------
salary       1 distinct answer(s): 82000 | 82000 | 82000
headcount    1 distinct answer(s): 3 | 3 | 3
combined     1 distinct answer(s): 143000 | 143000 | 143000
compare      1 distinct answer(s): Ana | Ana | Ana
----------------------------------------------------------------------


### 3. The smallest difference you can detect

With R runs and an observed standard deviation `sd`, the standard error of the mean
is `sd / sqrt(R)`. A rough two-sigma rule: to believe a difference between two
configurations, it should exceed about `2 * sqrt(2) * sd / sqrt(R)`.

This is back-of-envelope - the underlying distribution is binomial, not normal, and
R=3 is far too few for the normal approximation. Use it as an order-of-magnitude
guide, and note that on a 4-task benchmark **one task is 0.25 of the score**, so the
smallest difference you can even *express* is 0.25.

In [9]:
import math

sd = stats.stdev(scores) if R > 1 else 0.0
sem = sd / math.sqrt(R)
mde = 2 * math.sqrt(2) * sem
granularity = 1.0 / len(TASKS)

print("observed sd over %d runs        : %.3f" % (R, sd))
print("standard error of the mean     : %.3f" % sem)
print("rough detectable difference    : %.3f" % mde)
print("score granularity (1 task)     : %.3f" % granularity)
print()
print("So on THIS benchmark you cannot honestly claim any improvement smaller")
print("than %.3f." % max(mde, granularity))
print()
if sd == 0.0:
    print("The sd came out 0.000 - all %d runs scored identically. That does NOT" % R)
    print("mean the agent is deterministic; it means 3 runs of 4 easy tasks did")
    print("not sample any of the variance. The honest statement is: 'the noise")
    print("floor is below the resolution of this experiment', not 'there is no noise'.")

observed sd over 3 runs        : 0.000
standard error of the mean     : 0.000
rough detectable difference    : 0.000
score granularity (1 task)     : 0.250

So on THIS benchmark you cannot honestly claim any improvement smaller
than 0.250.

The sd came out 0.000 - all 3 runs scored identically. That does NOT
mean the agent is deterministic; it means 3 runs of 4 easy tasks did
not sample any of the variance. The honest statement is: 'the noise
floor is below the resolution of this experiment', not 'there is no noise'.


In [10]:
print("what it would take to detect a 5-percentage-point improvement")
print("-" * 62)
target = 0.05
for tasks_n, reps in [(4, 3), (20, 5), (100, 5), (200, 10)]:
    # Binomial standard error of a mean around p=0.75, over tasks_n*reps trials.
    p = 0.75
    se = math.sqrt(p * (1 - p) / (tasks_n * reps))
    detectable = 2 * math.sqrt(2) * se
    print("%3d tasks x %2d runs = %5d trials -> detectable diff %.3f  %s"
          % (tasks_n, reps, tasks_n * reps, detectable,
             "OK" if detectable <= target else "too noisy"))
print("-" * 62)
print("This is why serious agent benchmarks have hundreds of tasks. It is not")
print("thoroughness for its own sake - it is the sample size the claim requires.")

what it would take to detect a 5-percentage-point improvement
--------------------------------------------------------------
  4 tasks x  3 runs =    12 trials -> detectable diff 0.354  too noisy
 20 tasks x  5 runs =   100 trials -> detectable diff 0.122  too noisy
100 tasks x  5 runs =   500 trials -> detectable diff 0.055  too noisy
200 tasks x 10 runs =  2000 trials -> detectable diff 0.027  OK
--------------------------------------------------------------
This is why serious agent benchmarks have hundreds of tasks. It is not
thoroughness for its own sake - it is the sample size the claim requires.


### 4. Checklist for reporting an agent benchmark result

Everything here is cheap to record and impossible to reconstruct later:

- [ ] **Task set version** and task count
- [ ] **Model name and version string**, and the date
- [ ] **Temperature** and any sampling parameters
- [ ] **R** (repeats) and the **mean +/- sd**, not a single run
- [ ] **k and n** if you quote pass@k
- [ ] **Tokens and cost** per run
- [ ] The **per-task table**, so a reader can see which tasks moved
- [ ] Any task excluded, and why

If a benchmark result you are reading omits R, treat it as a single sample.

### Honest summary of this module

- Notebook 01 explained tau-bench, GAIA and SWE-bench conceptually. None of them
  uses an LLM judge for its headline metric - all three found a deterministic check.
- Notebook 02 built the same four pieces at 1/1000 scale and showed that a single
  score is a sample.
- Notebook 03 measured pass@1 and pass@3 with the unbiased estimator on n=3 samples
  per task, and was explicit that n=3 quantises pass@1 to {0, 1/3, 2/3, 1}.
- Notebook 04 measured the noise floor across R=3 identical runs, and computed that
  a 4-task benchmark cannot express a difference smaller than 0.25 regardless of how
  many times you run it.

The conclusion is not "our agent scores X". It is: **on a benchmark this small, no
score difference you observe is trustworthy** - and now you know how to compute how
big it would need to be.

### Where to go next

- [`../07_llm_as_judge`](../07_llm_as_judge) - for the tasks where no deterministic
  grader exists. Prefer a checker; use a judge when you must, and calibrate it.
- [`../06_multi_agent_debate`](../06_multi_agent_debate) - the n=4 comparison there
  is exactly the kind of result this module tells you not to trust. Re-run it through
  this harness with R repeats if you want a real answer.
- [`../../03_rag_advanced/10_rag_evaluation/05_agent_trajectory_evaluation.ipynb`](../../03_rag_advanced/10_rag_evaluation/05_agent_trajectory_evaluation.ipynb) -
  grading the *path* rather than the final answer.